# PCR — Priority-Conditioned Retention (test-time, Mamba-790m)
**Importance-Aware Capacity Allocation in a pretrained continuous-state SSM**

Training-free inference-time retention control on pretrained Mamba-790m.

- Scope: single model (790m). 1.4b failure and natural-language fragility are reported as limitations.
- Two levers: (1) survival via Δ gate, (2) precision via write quantization.
- Runs on T4 in tens of minutes. Do NOT install `mamba-ssm`/`causal-conv1d` — slow path required for Δ hooks.

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Jun 20 03:02:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# block fast path → force slow path so dt_proj/conv1d hooks fire
!pip -q install "transformers>=4.39" accelerate
import torch, random, math
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# load pretrained 790m, fp32, frozen
from transformers import AutoTokenizer, MambaForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "state-spaces/mamba-790m-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = MambaForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
NL = len(model.backbone.layers)
print("layers", NL)
print("note: 'fast path is not available ... sequential implementation' is expected — slow path confirmed")

In [ ]:
# per-position gain on dt_proj output (pre-softplus); controlled via _CTRL
_CTRL = {'gain_vec': None, 'layers': None}
def _mk_dt_hook(idx):
    def hook(mod, inp, out):
        g = _CTRL['gain_vec']
        if g is None: return out
        if _CTRL['layers'] is not None and idx not in _CTRL['layers']: return out
        L = out.shape[1]
        gv = g[:L].to(out.dtype).to(out.device)
        return out * gv.view(1, L, 1)
    return hook
_dt_handles = []
def setup_dt_hooks():
    global _dt_handles
    for h in _dt_handles: h.remove()
    _dt_handles = [l.mixer.dt_proj.register_forward_hook(_mk_dt_hook(i))
                   for i, l in enumerate(model.backbone.layers)]
def verify_dt_hooks():
    ids = tokenizer(" a b c d e f", return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad(): base = model(ids).logits[0, -1].clone()
    _CTRL['gain_vec'] = torch.full((ids.shape[1],), 0.5); _CTRL['layers'] = None
    with torch.no_grad(): pert = model(ids).logits[0, -1]
    _CTRL['gain_vec'] = None
    d = (base - pert).abs().max().item()
    return d > 1e-4, d
setup_dt_hooks()
ok, d = verify_dt_hooks(); print("Δ hook fires:", ok, "| max logit change", round(d, 3))

In [ ]:
# mean decay A (-exp(A_log)) per layer; A < 0 means recency bias
_A = [(-torch.exp(l.mixer.A_log)).mean().item() for l in model.backbone.layers]
print("mean A: L0", round(_A[0],3), "| Lmid", round(_A[NL//2],3), "| Llast", round(_A[-1],3))

In [ ]:
# single-token word pool + MQAR builder (each word = 1 token → clean position accounting)
def single_token_words(n, seed=0):
    rng = random.Random(seed); ids = list(range(1000, 45000)); rng.shuffle(ids); out = []
    for i in ids:
        s = tokenizer.decode([i]).strip()
        if (s.isascii() and s.isalpha() and 3 <= len(s) <= 8
                and len(tokenizer(" " + s, add_special_tokens=False).input_ids) == 1):
            out.append(s)
        if len(out) >= n: break
    return out
_WORDS = single_token_words(4000); print("single-token words:", len(_WORDS))

def _enc(word):
    return tokenizer(" " + word, add_special_tokens=False).input_ids

def build_mqar(N, rng, query_pair=0):
    ws = rng.sample(_WORDS, 2 * N)
    pairs = [(ws[2*j], ws[2*j+1]) for j in range(N)]
    seq = []; pos = []
    for (k, v) in pairs:
        kp = len(seq); seq += _enc(k)
        vp = len(seq); seq += _enc(v); pos.append((kp, vp))
    seq += tokenizer(" ?", add_special_tokens=False).input_ids
    seq += _enc(pairs[query_pair][0])
    ids = torch.tensor([seq], device=DEVICE)
    tgt = _enc(pairs[query_pair][1])[0]
    return ids, pos, tgt, pairs

def _recall_once(ids, tgt):
    with torch.no_grad(): lg = model(ids).logits[0, -1]
    return int(lg.argmax().item() == tgt)

In [ ]:
# capacity curve: find N where base recall ≈ 0.4
def base_recall(N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t)); h += _recall_once(ids, tgt)
    return h / n
print("[capacity curve]")
for N in [2, 8, 16, 24, 32, 48, 64]:
    print(f"  N={N:>3}: {base_recall(N, n=30):.2f}")

In [ ]:
# Gate1: downstream Δ suppression (og < 1) → target retention
def trials_delta(N, og, layers=None, n=30, seed0=0, scope='downstream'):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        L = ids.shape[1]; v0 = pos[0][1]; gv = torch.ones(L)
        if scope == 'downstream':
            gv[v0 + 1:] = og
        elif scope == 'random':
            idxs = list(range(v0 + 1, L)); random.Random(seed0 + t + 999).shuffle(idxs)
            for j in idxs[:max(1, len(idxs)//2)]: gv[j] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n

N = 32
b = base_recall(N); print("base", round(b, 2))
print("[Gate1 dose] og sweep (all layers)")
for og in [1.0, 0.9, 0.8, 0.7, 0.6, 0.4]:
    print(f"  og={og}: {trials_delta(N, og, n=30):.2f}")
print("[Gate1 specificity] downstream vs random (og=0.8)")
print(f"  downstream {trials_delta(N,0.8,scope='downstream',n=30):.2f} | random {trials_delta(N,0.8,scope='random',n=30):.2f}")

In [ ]:
# layer localization: which 8-layer band carries the recall gain?
groups = {f"L{a}-{a+7}": list(range(a, a + 8)) for a in range(0, NL, 8)}
base = base_recall(N)
print(f"[layer localization] og=0.8, base={base:.2f}")
for name, g in groups.items():
    r = trials_delta(N, 0.8, layers=g, n=30)
    print(f"  {name}: {r:.2f}  (Δ {r-base:+.2f})")

In [ ]:
# surgical L16-23 vs all-layer + ppl guardrail
def ppl_under_gain(og, layers=None):
    txt = ("The transformer relies on attention while state space models compress "
           "history into a fixed hidden state for linear time inference.")
    ids = tokenizer(txt, return_tensors='pt').input_ids.to(DEVICE); L = ids.shape[1]
    _CTRL['gain_vec'] = torch.full((L,), og); _CTRL['layers'] = set(layers) if layers else None
    with torch.no_grad(): nll = model(ids, labels=ids).loss.item()
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return math.exp(nll)
base_ppl = ppl_under_gain(1.0); L1623 = list(range(16, 24))
print(f"base ppl {base_ppl:.1f}")
print(f"  full   og=0.8: recall {trials_delta(N,0.8,n=30):.2f}  ppl x{ppl_under_gain(0.8)/base_ppl:.2f}")
print(f"  L16-23 og=0.8: recall {trials_delta(N,0.8,layers=L1623,n=30):.2f}  ppl x{ppl_under_gain(0.8,L1623)/base_ppl:.2f}")

In [ ]:
# write-order as implicit priority: recall by pair position
def recall_of_pair(N, j, og, layers=None, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, pairs = build_mqar(N, random.Random(seed0 + t), query_pair=j)
        L = ids.shape[1]; gv = torch.ones(L)
        if og < 1.0: gv[pos[j][1] + 1:] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n
print("[write-order priority] N=32, L16-23 og=0.8")
for j, lab in [(0, 'early(P1)'), (N//2, 'mid(P2)'), (N-1, 'late(P3)')]:
    bb = recall_of_pair(N, j, 1.0, n=30); aa = recall_of_pair(N, j, 0.8, layers=L1623, n=30)
    print(f"  {lab:>10} pair{j:>2}: base {bb:.2f} -> {aa:.2f}  (Δ {aa-bb:+.2f})")

In [ ]:
# graded P1/P2/P3: assign per-level og to each write position (exploratory)
def graded_recall(N, og_lv, layers=None, n=30, seed0=0):
    res = {1: 0, 2: 0, 3: 0}; cnt = {1: 0, 2: 0, 3: 0}
    for t in range(n):
        rng = random.Random(seed0 + t); ws = rng.sample(_WORDS, 2 * N)
        pairs = [(ws[2*i], ws[2*i+1]) for i in range(N)]
        lvl = [(i % 3) + 1 for i in range(N)]
        seq = []; pos = []
        for (k, v) in pairs:
            kp = len(seq); seq += _enc(k)
            vp = len(seq); seq += _enc(v); pos.append((kp, vp))
        for Lq in [1, 2, 3]:
            j = lvl.index(Lq)
            s2 = seq + tokenizer(" ?", add_special_tokens=False).input_ids + _enc(pairs[j][0])
            ids = torch.tensor([s2], device=DEVICE); tgt = _enc(pairs[j][1])[0]
            gv = torch.ones(ids.shape[1])
            for i, (kp, vp) in enumerate(pos):
                gv[vp] = og_lv[lvl[i]]
            _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
            res[Lq] += _recall_once(ids, tgt); cnt[Lq] += 1
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return {k: res[k] / cnt[k] for k in [1, 2, 3]}
g = graded_recall(N, {1: 0.7, 2: 0.85, 3: 1.0}, layers=L1623, n=30)
print("[graded P1/P2/P3] L16-23")
print(f"  P1 {g[1]:.2f} | P2 {g[2]:.2f} | P3 {g[3]:.2f}")
print("  monotone P1>P2>P3: graded allocation holds; broken: SSM uniform-decay limit")

In [ ]:
# lever 2: write precision — fake-quantize conv1d output at selected positions
def fake_quant_vec(v, bits):
    if bits >= 16: return v
    qmax = (2 ** bits) - 1; lo = v.amin(-1, keepdim=True); hi = v.amax(-1, keepdim=True)
    scale = (hi - lo).clamp(min=1e-8) / qmax
    return torch.round((v - lo) / scale) * scale + lo
_QMASK = {'pos': None, 'bits': 16, 'fired': 0}
def _qhook(mod, inp, out):
    if _QMASK['bits'] >= 16 or not _QMASK['pos']: return out
    L = out.shape[-1]; pos = [p for p in _QMASK['pos'] if 0 <= p < L]
    if not pos: return out
    o = out.clone()
    for p in pos: o[:, :, p] = fake_quant_vec(o[:, :, p], _QMASK['bits'])
    _QMASK['fired'] += 1; return o
_qh = [l.mixer.conv1d.register_forward_hook(_qhook) for l in model.backbone.layers]

def rd_target(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        _QMASK['pos'] = list(pos[0]); _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
def rd_distractor(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        dp = [p for pr in pos[1:] for p in pr]
        _QMASK['pos'] = dp; _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
print("[A] target k-bit → recall (rate-distortion curve)")
for bt in [16, 8, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_target(bt, N, n=30):.2f}")
print("[B] distractor k-bit → target recall (priority reallocation)")
for bt in [16, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_distractor(bt, N, n=30):.2f}")

In [ ]:
# exp A-1: 8-layer band localization across N=16/32/64
# does L16-23 remain the best band under varying capacity pressure?

sweep_Ns   = [16, 32, 64]
band_names = [f'L{a}-{a+7}' for a in range(0, NL, 8)]
band_layers = [list(range(a, a + 8)) for a in range(0, NL, 8)]

print('[exp A-1] band localization across N  (og=0.8)')
header = f"{'N':>4} | {'base':>5} | " + ' | '.join(f'{b:>9}' for b in band_names)
print(header)
print('-' * len(header))

nsweep_loc = {}
for N in sweep_Ns:
    n_trials = 20 if N == 64 else 30
    b = base_recall(N, n=n_trials)
    band_r = [trials_delta(N, 0.8, layers=bl, n=n_trials) for bl in band_layers]
    nsweep_loc[N] = {'base': b, 'bands': dict(zip(band_names, band_r))}
    cols = ' | '.join(f'{r:.2f}({r-b:+.2f})' for r in band_r)
    print(f'{N:>4} | {b:>5.2f} | {cols}')

print('\nbest band per N:')
for N, res in nsweep_loc.items():
    best = max(res['bands'], key=res['bands'].get)
    print(f'  N={N:>2}: {best}  recall={res["bands"][best]:.2f}  gain={res["bands"][best]-res["base"]:+.2f}')

In [ ]:
# exp A-2: surgical L16-23 vs full — recall across N

print('[exp A-2] surgical (L16-23) vs full  (og=0.8)')
print(f"{'N':>4} | {'base':>6} | {'full':>6} | {'Δfull':>6} | {'surg':>6} | {'Δsurg':>6}")
print('-' * 48)

for N in sweep_Ns:
    n_trials = 20 if N == 64 else 30
    base_r = base_recall(N, n=n_trials)
    full_r  = trials_delta(N, 0.8, n=n_trials)
    surg_r  = trials_delta(N, 0.8, layers=L1623, n=n_trials)
    print(f'{N:>4} | {base_r:>6.2f} | {full_r:>6.2f} | {full_r-base_r:>+6.2f} | '
          f'{surg_r:>6.2f} | {surg_r-base_r:>+6.2f}')

_base_ppl = ppl_under_gain(1.0)
_full_ppl = ppl_under_gain(0.8)
_surg_ppl = ppl_under_gain(0.8, L1623)
print(f'\nPPL (fixed passage, N-independent): '
      f'base={_base_ppl:.1f}  '
      f'full=×{_full_ppl/_base_ppl:.2f}  '
      f'surg(L16-23)=×{_surg_ppl/_base_ppl:.2f}')

## Notes

- **Capacity curve**: use N where base ≈ 0.4 as the operating point (N=32 from cell 7 onward).
- **Gate1**: recall peaks in a specific og range; downstream ≫ random confirms specificity.
- **Localization**: L16-23 is the peak band. Surgical L16-23 recovers most gain at PPL ×1.07 vs ×1.81 (all-layer).
- **Write-order**: early-pair Δ gain > late-pair → write order acts as implicit priority.
- **Graded P1/P2/P3**: monotone = graded allocation holds; non-monotone = SSM uniform-decay limit (both worth reporting).
- **Write quantization**: target-quant cliff at 4-bit; distractor-quant fails — negative result, superimposed state has no per-slot addressing.

**Limitations**: effects are robust on 790m synthetic MQAR but not replicated on 1.4b and fragile on natural language.